# New Roots 
##  Data Ingestion and Cleaning

This notebook loads our existing downloads and makes them simpler to work with. The next notebook will join the cleaned data and calculate the final suburb metrics (OBT).

In [55]:
# Install libraries 
%pip install pandas geopandas openpyxl pyogrio scipy numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [56]:
# Setup
from pathlib import Path
import zipfile
import re
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import numpy as np

#set the directories
datasets = Path("datasets")
cleaned = Path("cleaned")
cleaned.mkdir(exist_ok=True)

print("Datasets folder:", datasets.resolve())
print("Folder found:", datasets.exists())


def normalize_suburb_name(name):
    """Lowercase a suburb name and drop any trailing bracketed tag, e.g. the
    "(Vic.)" ABS adds to names that also exist in another state. Used to line
    up suburb names across tables that don't follow the ABS convention."""
    name = re.sub(r"\s*\([^)]*\)\s*$", "", str(name))
    return name.strip().lower()

Datasets folder: C:\Monash\ITO5002\drive-download-20261107T100712Z-1-001\New Rental Data\datasets
Folder found: True


## 1. Load the data


In [57]:
# 1 - Median weekly rents
# The published workbook has multiple sheets and headings, so read them as they appear.
file = next(datasets.rglob("moving_annual_rent_latest.xlsx"))
ds_01 = pd.read_excel(file, sheet_name=None, header=None)

print("Rental sheets:", list(ds_01))

Rental sheets: ['1 bedroom flat', '2 bedroom flat', '3 bedroom flat', '2 bedroom house', '3 bedroom house', '4 bedroom house', 'All properties']


In [58]:

# 1b - LGA rental data and suburb-to-LGA mapping

file = next(datasets.rglob(
    "moving_annual_rent_latest_LGA.xlsx"
))
ds_01_lga = pd.read_excel(file, sheet_name=None, header=None)

file = next(datasets.rglob("suburb_lga_mapping.csv"))
suburb_lga_mapping = pd.read_csv(file, dtype={"suburb_id": str})

print("LGA rental sheets:", list(ds_01_lga))
print("Suburbs mapped:", suburb_lga_mapping["suburb_id"].nunique())


LGA rental sheets: ['1 bedroom flat', '2 bedroom flat', '3 bedroom flat', '2 bedroom house', '3 bedroom house', '4 bedroom house', 'All properties']
Suburbs mapped: 572


In [59]:
# 2 - Criminal incidents by suburb

file = next(datasets.rglob("*LGA*Criminal*.xlsx"))

ds_02 = pd.read_excel(file, sheet_name="Table 03")

print("Crime records:", len(ds_02))
display(ds_02.head())

Crime records: 341869


,Year,Year ending,Local Government Area,Postcode,Suburb/Town Name,Offence Division,Offence Subdivision,Offence Subgroup,Incidents Recorded
0,2026,June,Alpine,3691,Coral Bank,A Crimes against the person,A20 Assault and related offences,A211 FV Serious assault,1
1,2026,June,Alpine,3691,Coral Bank,B Property and deception offences,B20 Property damage,B21 Criminal damage,1
2,2026,June,Alpine,3691,Dederang,A Crimes against the person,"A70 Stalking, harassment and threatening behav...",A732 Non-FV Threatening behaviour,1
3,2026,June,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
4,2026,June,Alpine,3691,Dederang,D Public order and security offences,D10 Weapons and explosives offences,D11 Firearms offences,1


In [60]:
# 3 - ABS Census 2021 (population and median age)
# Continue reading G01/G02 from the original Census DataPack ZIP.
file = next(datasets.rglob("2021_GCP_SAL_VIC.zip"))
ds_03 = {}

with zipfile.ZipFile(file) as z:
    for name in z.namelist():
        filename = Path(name).name.upper()
        if filename.endswith(".CSV") and any(f"2021CENSUS_G{t}" in filename for t in ["01_", "02_"]):
            with z.open(name) as csv_file:
                ds_03[Path(name).stem] = pd.read_csv(csv_file, low_memory=False)

print("Census tables:", list(ds_03))
assert any("G01_" in k.upper() for k in ds_03), "Census G01 is missing"
assert any("G02_" in k.upper() for k in ds_03), "Census G02 is missing"

# Detailed language-used-at-home data exported from 2021 Census TableBuilder.
# Save the supplied CSV anywhere within the datasets folder.
language_file = next(datasets.rglob("lanuage by suburb.csv"))
ds_03_languages = pd.read_csv(
    language_file, skiprows=9, skipfooter=8, engine="python", encoding="utf-8-sig"
)
print("TableBuilder language columns:", len(ds_03_languages.columns))

Census tables: ['2021Census_G01_VIC_SAL', '2021Census_G02_VIC_SAL']
TableBuilder language columns: 507


In [61]:
# 4 - ABS suburb boundaries

file = next(datasets.rglob("SAL_2021*.shp"), None)

if file is not None:
    ds_04 = gpd.read_file(file)
else:
    file = next((datasets / "04_sal_2021_boundaries").glob("*.zip"))
    with zipfile.ZipFile(file) as z:
        shp = next(name for name in z.namelist() if name.lower().endswith(".shp"))
    ds_04 = gpd.read_file(f"/vsizip/{file.resolve().as_posix()}/{shp}")

print("Suburb boundaries:", len(ds_04))
display(ds_04.head(3))

Suburb boundaries: 15353


,SAL_CODE21,SAL_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,SHAPE_Leng,SHAPE_Area,geometry
0,10001,Aarons Pass,1,New South Wales,AUS,Australia,82.7639,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.554241,0.007975,"POLYGON ((149.82477 -32.84384, 149.83271 -32.8..."
1,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485,"POLYGON ((150.86523 -33.88264, 150.86479 -33.8..."
2,10003,Abbotsford (NSW),1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099,"POLYGON ((151.13472 -33.85492, 151.13445 -33.8..."


In [62]:
# 5 - Greater Melbourne boundary
# Shapefile 
file = next(datasets.rglob("*GCCSA*.shp"))
ds_05 = gpd.read_file(file)

print("Regions:", len(ds_05))
display(ds_05.head(3))

Regions: 35


,GCC_CODE21,GCC_NAME21,CHG_FLAG21,CHG_LBL21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geometry
0,1GSYD,Greater Sydney,0,No change,1,New South Wales,AUS,Australia,12368.6858,http://linked.data.gov.au/dataset/asgsed3/GCCS...,"MULTIPOLYGON (((151.28159 -33.83318, 151.28151..."
1,1RNSW,Rest of NSW,0,No change,1,New South Wales,AUS,Australia,788428.9733,http://linked.data.gov.au/dataset/asgsed3/GCCS...,"MULTIPOLYGON (((159.0623 -31.50886, 159.06218 ..."
2,19499,No usual address (NSW),0,No change,1,New South Wales,AUS,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/GCCS...,None


In [63]:
# 6 - Vicmap Features of Interest
file = next(datasets.rglob("FOI_POINT.shp"))
ds_06 = gpd.read_file(file)

print("Vicmap locations:", len(ds_06))
display(ds_06.head(3))

Vicmap locations: 52118


,UFI,PFI,FEATURE_ID,PARENTFTID,FTYPE,FEATSUBTYP,FEATSTATUS,NAME,NAME_LABEL,PARENTNAME,...,VICNMSTATC,THEME1,THEME2,STATE,CRDATE_PFI,SUPER_PFI,CRDATE_UFI,FEATURE_UF,FEATURE_CR,geometry
0,45544976,984269,984269,984268.0,education centre,tertiary institution,None,CHISHOLM INSTITUTE OF TAFE - WONTHAGGI CAMPUS,Chisholm Institute Of Tafe - Wonthaggi Campus,CHISHOLM INSTITUTE OF TAFE,...,11,None,None,VIC,2012-08-24,NaN,2013-07-30,45544976,2013-07-30,MULTIPOINT ((145.58204 -38.59086))
1,45545434,136319,136319,NaN,education centre,tertiary institution,None,CATHEDRAL COLLEGE,Cathedral College,NaN,...,11,None,None,VIC,2009-05-21,NaN,2013-07-30,45545434,2013-07-30,MULTIPOINT ((146.32915 -36.3853))
2,45544409,634128,634128,634149.0,education centre,tertiary institution,None,GORDON INSTITUTE OF TAFE - EAST GEELONG CAMPUS,Gordon Institute Of Tafe - East Geelong Campus,GORDON INSTITUTE OF TAFE,...,11,None,None,VIC,2009-08-31,NaN,2013-07-30,45544409,2013-07-30,MULTIPOINT ((144.38308 -38.174))


In [64]:
# 7 - Public transport stops
# We do not need the much larger transport lines file for these metrics.
file = next(datasets.rglob("public_transport_stops.geojson"))
ds_07 = gpd.read_file(file)

print("Transport stops:", len(ds_07))
display(ds_07.head(3))

Transport stops: 32063


,STOP_ID,STOP_NAME,MODE,geometry
0,11212,Flinders Street Station,REGIONAL TRAIN,POINT (144.96627 -37.81809)
1,11213,Flinders Street Station,REGIONAL TRAIN,POINT (144.96649 -37.81814)
2,11214,Flinders Street Station,REGIONAL TRAIN,POINT (144.96652 -37.8182)


In [65]:
# 10 - Hospitals, GPs and pharmacies
# Each service type is stored in its own GeoJSON file.
folder = datasets / "10_health_services"
ds_10 = {}

for file in sorted(folder.glob("*.geojson")):
    ds_10[file.stem] = gpd.read_file(file)

print("Health datasets:", list(ds_10))
for name, table in ds_10.items():
    print(name, len(table))

Health datasets: ['general_practices', 'hospitals', 'pharmacies']
general_practices 8082
hospitals 877
pharmacies 5622


## 2. Clean and simplify the tables


In [66]:
# 4 & 5 - Suburbs in Greater Melbourne
# Keep suburb code, name, area and boundary for our future OBT.
melbourne = ds_05[ds_05["GCC_NAME21"].str.lower() == "greater melbourne"].to_crs(7855)
assert len(melbourne) == 1, "Expected one Greater Melbourne boundary"
melbourne_shape = melbourne.geometry.union_all()

vic_suburbs = ds_04[ds_04["STE_CODE21"].astype(str).str.lstrip("0") == "2"].to_crs(7855)
suburbs = vic_suburbs[vic_suburbs.geometry.representative_point().within(melbourne_shape)].copy()
suburbs = suburbs[["SAL_CODE21", "SAL_NAME21", "AREASQKM21", "geometry"]]
suburbs = suburbs.rename(columns={
    "SAL_CODE21": "suburb_id", "SAL_NAME21": "suburb_name", "AREASQKM21": "suburb_area_km2"
})
suburbs["suburb_id"] = suburbs["suburb_id"].astype(str)
suburbs["suburb_area_km2"] = pd.to_numeric(suburbs["suburb_area_km2"], errors="coerce")

# normalised name for joining to crime/rental later, since they don't use ABS's "(Vic.)" tag
suburbs["suburb_name_join"] = suburbs["suburb_name"].map(normalize_suburb_name)
suburbs = suburbs.to_crs(4326).reset_index(drop=True)

print("Greater Melbourne suburbs:", len(suburbs))
display(suburbs.head())

# JG observation: ABS tags suburb names that also exist in another state with "(Vic.)",
# e.g. Abbotsford (Vic.). Nothing else in our other sources does this, so a plain name
# join against crime/rental would miss most suburbs. Added suburb_name_join above to fix that.

Greater Melbourne suburbs: 572


,suburb_id,suburb_name,suburb_area_km2,geometry,suburb_name_join
0,20002,Abbotsford (Vic.),1.7405,"POLYGON ((145.00195 -37.79665, 145.0019 -37.79...",abbotsford
1,20003,Aberfeldie,1.5515,"POLYGON ((144.89576 -37.76514, 144.89547 -37.7...",aberfeldie
2,20011,Aintree,6.7302,"POLYGON ((144.67179 -37.72628, 144.67022 -37.7...",aintree
3,20015,Airport West,3.6748,"POLYGON ((144.87979 -37.71565, 144.88016 -37.7...",airport west
4,20017,Albanvale,1.8634,"POLYGON ((144.76926 -37.74082, 144.77088 -37.7...",albanvale


In [67]:
# 2 - Clean and transform crime data

# Keep the latest year
crime = ds_02[ds_02["Year"] == 2026].copy()

# Keep only the columns needed
crime = crime[[
    "Local Government Area",
    "Postcode",
    "Suburb/Town Name",
    "Offence Division",
    "Incidents Recorded"
]]

# Rename columns
crime.columns = ["lga", "postcode", "suburb", "offence_type", "incidents"]

# Convert incident counts to numbers
crime["incidents"] = pd.to_numeric(crime["incidents"], errors="coerce")

# Separate crimes against people and property offences
crime["person_incidents"] = crime["incidents"].where(
    crime["offence_type"].str.startswith("A "), 0
)

crime["property_incidents"] = crime["incidents"].where(
    crime["offence_type"].str.startswith("B "), 0
)

# Combine records into one row per suburb, postcode and LGA
crime_clean = crime.groupby(
    ["lga", "postcode", "suburb"], as_index=False
)[["incidents", "person_incidents", "property_incidents"]].sum()

# Rename the total incidents column
crime_clean = crime_clean.rename(columns={
    "incidents": "total_incidents"
})

# matching key for suburbs.suburb_name_join
crime_clean["suburb_join"] = crime_clean["suburb"].map(normalize_suburb_name)

# Crime data is reported statewide, not just for Melbourne, so drop anything that
# isn't one of our 572 suburbs - same scoping the suburbs/vicmap/transport/health
# tables already apply, just done by name instead of by shape.
crime_clean = crime_clean[crime_clean["suburb_join"].isin(suburbs["suburb_name_join"])]

print("Crime records:", len(crime_clean))
display(crime_clean.head())

# JG observation: joining straight on suburb name only matched ~20% of our suburbs. Turned out
# to be the ABS "(Vic.)" tag again - once that's stripped (see suburb_join), 569 of our 572
# Melbourne suburbs match. The remaining 3 just don't seem to have any recorded incidents.
# Also dropped every row that isn't one of our 572 suburbs - this data covers the whole
# state, so most of it was for towns well outside Melbourne and not relevant to the OBT.

Crime records: 648


,lga,postcode,suburb,total_incidents,person_incidents,property_incidents,suburb_join
113,Banyule,3079,Ivanhoe,922,114,697,ivanhoe
114,Banyule,3079,Ivanhoe East,123,5,96,ivanhoe east
115,Banyule,3081,Bellfield,127,20,86,bellfield
116,Banyule,3081,Heidelberg Heights,553,94,343,heidelberg heights
117,Banyule,3081,Heidelberg West,998,157,612,heidelberg west


In [68]:
# 3 - Census population and median age
# G01 and G02 have one row per suburb. Give the fields shorter names.
g01 = next(table for name, table in ds_03.items() if "G01_" in name.upper())
g02 = next(table for name, table in ds_03.items() if "G02_" in name.upper())

code_1 = next(c for c in g01.columns if "SAL_CODE" in c.upper())
code_2 = next(c for c in g02.columns if "SAL_CODE" in c.upper())
pop_col = "Tot_P_P"
age_col = next(c for c in g02.columns if "med" in c.lower() and "age" in c.lower())

population = g01[[code_1, pop_col]].rename(columns={code_1: "suburb_id", pop_col: "population"})
ages = g02[[code_2, age_col]].rename(columns={code_2: "suburb_id", age_col: "median_age"})

# ABS census codes come through as e.g. SAL20001 - strip the letters so they match
# the plain numeric codes used in the suburb boundary file
population["suburb_id"] = population["suburb_id"].astype(str).str.replace(r"^\D+", "", regex=True)
ages["suburb_id"] = ages["suburb_id"].astype(str).str.replace(r"^\D+", "", regex=True)
census_people = population.merge(ages, on="suburb_id", how="left")
census_people["population"] = pd.to_numeric(census_people["population"], errors="coerce")
census_people["median_age"] = pd.to_numeric(census_people["median_age"], errors="coerce")
census_people = census_people[census_people["suburb_id"].isin(suburbs["suburb_id"])]

print("Median age source column:", age_col)
display(census_people.head())

# JG observation: this came back empty at first - the census suburb codes have a "SAL" prefix
# that the boundary file's codes don't, so they never matched until stripped.

Median age source column: Median_age_persons


,suburb_id,population,median_age
1,20002,9088,33
2,20003,3925,41
10,20011,7982,30
14,20015,8173,39
16,20017,5641,37


In [69]:
# 3 - Census language used at home (2021 TableBuilder, detailed LANP)
# The TableBuilder CSV is wide: one suburb per row and one column per language.
language_wide = ds_03_languages.copy()
language_wide = language_wide.rename(columns={language_wide.columns[0]: "suburb_name"})
language_wide["suburb_name"] = language_wide["suburb_name"].astype("string").str.strip()

# Remove the extra header, state total, non-resident records, and unused columns.
language_wide = language_wide[
    language_wide["suburb_name"].notna()
    & ~language_wide["suburb_name"].isin([
        "SAL (UR)", "Total", "No usual address (Vic.)",
        "Migratory - Offshore - Shipping (Vic.)"
    ])
].copy()
language_columns = [
    c for c in language_wide.columns
    if c not in {"suburb_name", "Total", "Not stated"}
    and not c.startswith("Unnamed:")
]

# Keep detailed named languages as well as explicitly labelled nfd/nec categories.
# Reject missing/invalid counts rather than silently treating them as zero.
language_numeric = language_wide[language_columns].apply(pd.to_numeric, errors="coerce")
language_wide = pd.concat([language_wide[["suburb_name"]], language_numeric], axis=1)
if language_wide[language_columns].isna().any().any():
    raise ValueError("TableBuilder language counts contain blank or nonnumeric values")
if language_wide[language_columns].lt(0).any().any():
    raise ValueError("TableBuilder language counts contain negative values")
if language_wide["suburb_name"].duplicated().any():
    raise ValueError("TableBuilder has duplicate suburb names")

# Join to Greater Melbourne suburbs: use exact ABS names first so places with
# the same normalised name cannot accidentally be assigned the wrong suburb ID.
suburb_lookup = suburbs[["suburb_id", "suburb_name", "suburb_name_join"]].copy()
if suburb_lookup["suburb_name"].duplicated().any():
    raise ValueError("Suburbs table has duplicate suburb names")
language_wide["suburb_name_join"] = language_wide["suburb_name"].map(normalize_suburb_name)
matched_exact = language_wide.merge(
    suburb_lookup[["suburb_id", "suburb_name"]], on="suburb_name",
    how="inner", validate="one_to_one"
)

# For names with different formatting, use the existing name normalisation,
# but only if the normalised name is unique in BOTH tables.
source_left = language_wide[
    ~language_wide["suburb_name"].isin(matched_exact["suburb_name"])
]
suburbs_left = suburb_lookup[
    ~suburb_lookup["suburb_id"].isin(matched_exact["suburb_id"])
]
source_left = source_left[
    ~source_left["suburb_name_join"].duplicated(keep=False)
]
suburbs_left = suburbs_left[
    ~suburbs_left["suburb_name_join"].duplicated(keep=False)
]
matched_fallback = source_left.merge(
    suburbs_left[["suburb_id", "suburb_name_join"]],
    on="suburb_name_join", how="inner", validate="one_to_one"
)
matched = pd.concat([matched_exact, matched_fallback], ignore_index=True)
if matched["suburb_id"].duplicated().any():
    raise ValueError("More than one language record matched the same suburb ID")

unmatched_suburbs = suburb_lookup[
    ~suburb_lookup["suburb_id"].isin(matched["suburb_id"])
]
print(f"TableBuilder suburbs matched: {len(matched)} / {len(suburbs)}")
if len(unmatched_suburbs):
    print("Unmatched Greater Melbourne suburbs (first 20):")
    display(unmatched_suburbs[["suburb_id", "suburb_name"]].head(20))

# One row for each suburb and language, ready for the app's dynamic lookup.
language_counts = matched.melt(
    id_vars="suburb_id", value_vars=language_columns,
    var_name="language", value_name="speaker_count"
)
language_counts["speaker_count"] = language_counts["speaker_count"].astype("int64")

print("Detailed language/classification categories:", len(language_columns))
print("Language count rows:", len(language_counts))
display(language_counts.head())

TableBuilder suburbs matched: 572 / 572
Detailed language/classification categories: 503
Language count rows: 287716


,suburb_id,language,speaker_count
0,20002,"Northern European Languages, nfd",0
1,20003,"Northern European Languages, nfd",0
2,20011,"Northern European Languages, nfd",0
3,20015,"Northern European Languages, nfd",0
4,20017,"Northern European Languages, nfd",0


In [70]:
# 1 - Clean and transform rental data

rent_tables = []

for property_type, sheet in ds_01.items():

    # Find the latest median rent column
    median_cols = [
        col for col in sheet.columns
        if str(sheet.loc[2, col]).strip().lower() == "median"
    ]

    latest_col = median_cols[-1]
    latest_period = sheet.loc[1, latest_col]

    # Keep suburb and latest median rent
    rent = sheet.loc[3:, [1, latest_col]].copy()
    rent.columns = ["suburb", "median_weekly_rent"]

    # Convert rent to a number and remove empty records
    rent["median_weekly_rent"] = pd.to_numeric(
        rent["median_weekly_rent"], errors="coerce"
    )

    rent = rent.dropna(subset=["suburb"])

    # The report is laid out in regional blocks, each closed off with a "Group Total"
    # row - that's a subtotal, not a suburb, so drop it before it gets treated as one.
    rent = rent[rent["suburb"].str.strip().str.lower() != "group total"]

    # Add dwelling type and reporting period
    rent["dwelling_type"] = property_type
    rent["period"] = latest_period

    rent_tables.append(rent)

# Combine all dwelling types into one table
rental_clean = pd.concat(rent_tables, ignore_index=True)

# Most rows aren't a single suburb but a combined precinct, e.g. "Carlton-Parkville".
# Keep the original label and also split it out so each suburb named in a precinct
# gets its own row carrying that precinct's rent figure.
rental_clean["precinct"] = rental_clean["suburb"]
rental_clean["suburb"] = rental_clean["suburb"].str.split("-")
rental_clean = rental_clean.explode("suburb", ignore_index=True)
rental_clean["suburb"] = rental_clean["suburb"].str.strip()


def normalize_rental_suburb_name(name):
    """Matching key just for lining up rental suburb names against `suburbs`.
    Only rental needs this extra cleanup: REIV sometimes writes compound names
    in a different order to ABS ("East Brunswick" vs "Brunswick East") and
    abbreviates "Mount" to "Mt", so on top of the usual lowercase/"(Vic.)" strip
    this also expands "Mt" and sorts the words."""
    name = normalize_suburb_name(name)
    name = re.sub(r"\bmt\b", "mount", name)
    return " ".join(sorted(name.split()))


# Drop the ABS "(Vic.)" tag from the suburbs list, then use that as the source of
# truth: wherever a rental suburb name matches one, overwrite it with the ABS
# spelling so both tables end up using exactly the same suburb names.
suburb_name_clean = suburbs["suburb_name"].str.replace(r"\s*\([^)]*\)\s*$", "", regex=True).str.strip()
canonical_lookup = dict(zip(suburb_name_clean.map(normalize_rental_suburb_name), suburb_name_clean))
rental_clean["suburb"] = rental_clean["suburb"].map(
    lambda name: canonical_lookup.get(normalize_rental_suburb_name(name), name)
)
rental_clean["suburb_join"] = rental_clean["suburb"].map(normalize_suburb_name)

print("Rental records:", len(rental_clean))
display(rental_clean.head())


# Clean LGA rental data using the same approach as suburb rents

lga_rent_tables = []

for property_type, sheet in ds_01_lga.items():


    # Find the latest median rent column
    median_cols = [
        col for col in sheet.columns
        if str(sheet.loc[2, col]).strip().lower() == "median"
    ]

    latest_col = median_cols[-1]
    latest_period = sheet.loc[1, latest_col]

    # Keep LGA and latest median rent
    rent = sheet.loc[3:, [1, latest_col]].copy()
    rent.columns = ["lga_name", "median_weekly_rent"]

    rent["median_weekly_rent"] = pd.to_numeric(
        rent["median_weekly_rent"], errors="coerce"
    )

    rent = rent.dropna(subset=["lga_name", "median_weekly_rent"])

    # Remove totals and clean names
    rent = rent[rent["lga_name"].str.strip().str.lower() != "group total"]


    rent["dwelling_type"] = property_type
    rent["period"] = latest_period

    lga_rent_tables.append(rent)

lga_rental_clean = pd.concat(lga_rent_tables, ignore_index=True)

# Create a matching key for LGA names
lga_rental_clean["lga_join"] = (lga_rental_clean["lga_name"].str.lower().str.strip())

print("LGA rental records:", len(lga_rental_clean))


# JG observation: about 1 in 12 rows here used to be "Group Total" - a region subtotal from
# the source spreadsheet, not an actual suburb - and were silently being counted as one.
# Dropped those, then split the combined precinct names (e.g. Carlton-Parkville) so each
# suburb in a precinct carries that precinct's rent figure. REIV also sometimes writes
# compound suburb names in a different order to ABS ("East Brunswick" vs "Brunswick East")
# or abbreviates "Mount" to "Mt" - normalize_rental_suburb_name (above) handles both, just
# for this table, and matched names get overwritten with the ABS spelling so the two tables
# agree. Altogether that lifts real suburb coverage from about 10% to around 29%, which
# looks like the practical ceiling: REIV only reports rent for ~146 named areas across the
# whole state, so most smaller Melbourne suburbs just aren't in this dataset at all - no
# amount of name-cleaning will recover that.

Rental records: 1505


,suburb,median_weekly_rent,dwelling_type,period,precinct,suburb_join
0,Albert Park,480.0,1 bedroom flat,Jun 2026,Albert Park-Middle Park-West St Kilda,albert park
1,Middle Park,480.0,1 bedroom flat,Jun 2026,Albert Park-Middle Park-West St Kilda,middle park
2,St Kilda West,480.0,1 bedroom flat,Jun 2026,Albert Park-Middle Park-West St Kilda,st kilda west
3,Armadale,478.0,1 bedroom flat,Jun 2026,Armadale,armadale
4,Carlton North,465.0,1 bedroom flat,Jun 2026,Carlton North,carlton north


LGA rental records: 480


In [71]:

# 1b - Five-year average new rentals with spatial density infill


NEAREST_SUBURBS = 5
YEARS_TO_AVERAGE = 5

# Identify the latest June reporting year from the rental data
sheet = next(iter(ds_01.values()))

june_years = []

for col in sheet.columns:
    if str(sheet.loc[2, col]).strip().lower() != "count":
        continue

    period = pd.to_datetime(
        str(sheet.loc[1, col]), format="%b %Y", errors="coerce"
    )

    if pd.notna(period) and period.month == 6:
        june_years.append(period.year)

latest_year = max(june_years)
YEARS = range(latest_year - YEARS_TO_AVERAGE + 1, latest_year + 1)

print("Rental availability years:", list(YEARS))

DWELLING_TYPES = [
    "1 bedroom flat",
    "2 bedroom flat",
    "3 bedroom flat",
    "2 bedroom house",
    "3 bedroom house",
    "4 bedroom house"
]

# Keep suburb geometry and calculate representative points
rental_suburbs = suburbs[
    ["suburb_id", "suburb_name_join", "geometry"]
].to_crs(7855).copy()

rental_suburbs["suburb_area_km2"] = (
    rental_suburbs.geometry.area / 1_000_000
)

rental_suburbs["x"] = (
    rental_suburbs.geometry.representative_point().x
)
rental_suburbs["y"] = (
    rental_suburbs.geometry.representative_point().y
)

# Extract five annual rental counts from Homes Victoria
rental_activity_tables = []

for dwelling_type in DWELLING_TYPES:

    sheet = ds_01[dwelling_type]
    count_cols = []

    for year in YEARS:
        matches = [
            col for col in sheet.columns
            if str(sheet.loc[1, col]).strip() == f"Jun {year}"
            and str(sheet.loc[2, col]).strip().lower() == "count"
        ]

        assert len(matches) == 1, (
            f"Expected one Jun {year} count for {dwelling_type}"
        )

        count_cols.append(matches[0])

    activity = sheet.loc[3:, [1] + count_cols].copy()

    year_cols = [
        f"new_rentals_{year}" for year in YEARS
    ]

    activity.columns = ["precinct"] + year_cols

    activity = activity.dropna(subset=["precinct"])
    activity = activity[
        activity["precinct"].astype(str).str.strip().str.lower()
        != "group total"
    ].copy()

    activity[year_cols] = activity[year_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    # Calculate the average only where all five years are available
    activity["5yr_avg_new_rentals"] = (
        activity[year_cols].mean(axis=1)
    )

    activity.loc[
        activity[year_cols].count(axis=1) != 5,
        "5yr_avg_new_rentals"
    ] = np.nan

    activity["dwelling_type"] = dwelling_type
    rental_activity_tables.append(activity)

rental_activity = pd.concat(
    rental_activity_tables, ignore_index=True
)

assert not rental_activity.duplicated(
    ["precinct", "dwelling_type"]
).any(), "Duplicate rental precinct records"

# Reuse the existing rental precinct-to-suburb matching
precinct_mapping = rental_clean[
    ["precinct", "suburb_join", "dwelling_type"]
].drop_duplicates()

precinct_mapping = precinct_mapping[
    precinct_mapping["dwelling_type"].isin(DWELLING_TYPES)
]

precinct_mapping = precinct_mapping.merge(
    rental_suburbs[
        ["suburb_id", "suburb_name_join", "suburb_area_km2"]
    ],
    left_on="suburb_join",
    right_on="suburb_name_join",
    how="left"
)

# Check that all constituent suburbs are matched
precinct_mapping["matched"] = (
    precinct_mapping["suburb_id"].notna()
    & precinct_mapping["suburb_area_km2"].gt(0)
)

precinct_quality = precinct_mapping.groupby(
    ["precinct", "dwelling_type"], as_index=False
).agg(
    members=("suburb_join", "nunique"),
    members_matched=("matched", "sum"),
    precinct_area_km2=(
        "suburb_area_km2",
        lambda x: x.sum(min_count=1)
    )
)

precinct_quality["complete"] = (
    precinct_quality["members"].eq(
        precinct_quality["members_matched"]
    )
    & precinct_quality["precinct_area_km2"].gt(0)
)

# Allocate grouped precinct counts proportionally by suburb area
allocated = precinct_mapping.merge(
    precinct_quality[
        ["precinct", "dwelling_type",
         "precinct_area_km2", "complete"]
    ],
    on=["precinct", "dwelling_type"],
    how="left"
)

allocated = allocated.merge(
    rental_activity[
        ["precinct", "dwelling_type", "5yr_avg_new_rentals"]
    ],
    on=["precinct", "dwelling_type"],
    how="left"
)

allocated["allocated_rentals"] = (
    allocated["5yr_avg_new_rentals"]
    * allocated["suburb_area_km2"]
    / allocated["precinct_area_km2"]
)

allocated.loc[
    ~allocated["complete"], "allocated_rentals"
] = np.nan

allocated = allocated.dropna(
    subset=["suburb_id", "allocated_rentals"]
)

assert not allocated.duplicated(
    ["suburb_id", "dwelling_type"]
).any(), "Duplicate suburb/dwelling type mappings"

# Include all suburbs and six dwelling types
all_combinations = rental_suburbs[
    ["suburb_id", "suburb_area_km2", "x", "y"]
].merge(
    pd.DataFrame({"dwelling_type": DWELLING_TYPES}),
    how="cross"
)

rental_availability = all_combinations.merge(
    allocated[
        ["suburb_id", "dwelling_type",
         "allocated_rentals", "precinct"]
    ],
    on=["suburb_id", "dwelling_type"],
    how="left"
)

rental_availability = rental_availability.rename(
    columns={"allocated_rentals": "5yr_avg_new_rentals"}
)

# Record which values are observed or estimated
rental_availability["data_source"] = pd.NA

rental_availability.loc[
    rental_availability["5yr_avg_new_rentals"].notna(),
    "data_source"
] = "observed_precinct_allocated"

# Spatial density infill for each dwelling type
for dwelling_type in DWELLING_TYPES:

    # Only observed data can be used to estimate missing values
    donors = rental_availability[
        (rental_availability["dwelling_type"] == dwelling_type)
        & (
            rental_availability["data_source"]
            == "observed_precinct_allocated"
        )
        & rental_availability["suburb_area_km2"].gt(0)
    ].copy()

    targets = rental_availability[
        (rental_availability["dwelling_type"] == dwelling_type)
        & rental_availability["5yr_avg_new_rentals"].isna()
        & rental_availability["suburb_area_km2"].gt(0)
    ].copy()

    donors = donors.dropna(subset=["x", "y"])
    targets = targets.dropna(subset=["x", "y"])

    if donors.empty or targets.empty:
        continue

    # Annual new rentals per km2 for observed suburbs
    donor_density = (
        donors["5yr_avg_new_rentals"]
        / donors["suburb_area_km2"]
    ).to_numpy()

    # Find the five closest observed suburbs
    tree = cKDTree(
        donors[["x", "y"]].to_numpy()
    )

    k = min(NEAREST_SUBURBS, len(donors))

    _, idx = tree.query(
        targets[["x", "y"]].to_numpy(),
        k=k
    )

    idx = np.asarray(idx).reshape(len(targets), k)

    # Average density of the nearest suburbs
    estimated_density = donor_density[idx].mean(axis=1)

    # Estimate annual new rentals using target suburb area
    rental_availability.loc[
        targets.index, "5yr_avg_new_rentals"
    ] = (
        estimated_density
        * targets["suburb_area_km2"].to_numpy()
    )

    rental_availability.loc[
        targets.index, "data_source"
    ] = "spatial_infill"

# Keep diagnostic information separately
rental_availability_diagnostics = rental_availability[
    [
        "suburb_id", "dwelling_type", "precinct",
        "suburb_area_km2", "5yr_avg_new_rentals",
        "data_source"
    ]
].copy()

# Identify how each rental availability value was calculated
rental_availability["infilling_method"] = (
    rental_availability["data_source"].replace({
        "observed_precinct_allocated": "suburb",
        "spatial_infill": "proximity"
    })
)
_

# Final Silver table
rental_availability["5yr_avg_new_rentals"] = (
    rental_availability["5yr_avg_new_rentals"].round(1)
)

rental_availability = rental_availability[
    ["suburb_id", "dwelling_type", "5yr_avg_new_rentals",
     "infilling_method"]
]

print("Rental availability records:", len(rental_availability))
display(rental_availability.head())


Rental availability years: [2022, 2023, 2024, 2025, 2026]
Rental availability records: 3432


,suburb_id,dwelling_type,5yr_avg_new_rentals,infilling_method
0,20002,1 bedroom flat,413.6,suburb
1,20002,2 bedroom flat,463.2,suburb
2,20002,3 bedroom flat,49.8,suburb
3,20002,2 bedroom house,63.0,suburb
4,20002,3 bedroom house,41.2,suburb


In [72]:

# Check rental availability and spatial infill

print("Total records:", len(rental_availability))
print("Suburbs:", rental_availability["suburb_id"].nunique())
print("Dwelling types:", rental_availability["dwelling_type"].nunique())

print("Missing values:",
      rental_availability["5yr_avg_new_rentals"].isna().sum())

print("Duplicate records:",
      rental_availability.duplicated(
          ["suburb_id", "dwelling_type"]
      ).sum())

# Check observed versus infilled records
print("\nData source breakdown:")
print(
    rental_availability_diagnostics["data_source"]
    .value_counts(dropna=False)
)

# Summary by dwelling type
coverage = rental_availability.groupby("dwelling_type").agg(
    available=("5yr_avg_new_rentals", "count"),
    median=("5yr_avg_new_rentals", "median"),
    mean=("5yr_avg_new_rentals", "mean"),
    maximum=("5yr_avg_new_rentals", "max")
).reset_index()

display(coverage)


Total records: 3432
Suburbs: 572
Dwelling types: 6
Missing values: 0
Duplicate records: 0

Data source breakdown:
data_source
spatial_infill                 2480
observed_precinct_allocated     952
Name: count, dtype: int64


,dwelling_type,available,median,mean,maximum
0,1 bedroom flat,572,51.70,109.929545,2622.6
1,2 bedroom flat,572,143.45,248.920105,3227.0
2,2 bedroom house,572,31.75,58.122378,497.6
3,3 bedroom flat,572,69.85,133.131643,1729.5
4,3 bedroom house,572,182.60,580.785839,9679.2
5,4 bedroom house,572,99.00,581.081294,11887.2


In [73]:
# 5b - Infill missing rental data

# Ensure all suburbs have a space for rental for each dwelling type

# Ensure every spatial suburb has a row for each dwelling type

all_combinations = (
    suburbs[["suburb_name_join"]]
    .drop_duplicates()
    .rename(columns={"suburb_name_join": "suburb_join"})
    .merge(
        pd.DataFrame({
            "dwelling_type": rental_clean["dwelling_type"].unique()
        }),
        how="cross"
    )
)

rental_clean = all_combinations.merge(
    rental_clean,
    on=["suburb_join", "dwelling_type"],
    how="left"
)

rental_clean["suburb"] = rental_clean["suburb"].fillna(
    rental_clean["suburb_join"]
)

# Fill missing suburb rents using weighted LGA rents, then proximity

# Label original rental values
rental_clean["infilling_method"] = "suburb"
rental_clean.loc[
    rental_clean["median_weekly_rent"].isna(), "infilling_method"
] = pd.NA

# Link rental suburbs to suburb IDs
suburb_lookup = suburbs[["suburb_id", "suburb_name_join"]]

rental_clean = rental_clean.merge(
    suburb_lookup,
    left_on="suburb_join",
    right_on="suburb_name_join",
    how="left"
).drop(columns="suburb_name_join")

# Calculate weighted LGA rents
lga_mapping = suburb_lga_mapping.merge(
    lga_rental_clean,
    left_on="lga_name",
    right_on="lga_name",
    how="left"
)

lga_mapping["weighted_rent"] = (
    lga_mapping["median_weekly_rent"] * lga_mapping["lga_weighting"]
)

lga_rents = (
    lga_mapping.dropna(subset=["median_weekly_rent"])
    .groupby(["suburb_id", "dwelling_type"], as_index=False)
    .agg(
        weighted_rent=("weighted_rent", "sum"),
        total_weight=("lga_weighting", "sum")
    )
)

# Account for missing data if suburb is part of more than one LGA

lga_rents["lga_rent"] = (
    lga_rents["weighted_rent"] / lga_rents["total_weight"]
)

rental_clean = rental_clean.merge(
    lga_rents[["suburb_id", "dwelling_type", "lga_rent"]],
    on=["suburb_id", "dwelling_type"],
    how="left"
)

# Fill missing values from LGA
missing = rental_clean["median_weekly_rent"].isna() & rental_clean["lga_rent"].notna()

rental_clean.loc[missing, "median_weekly_rent"] = rental_clean.loc[missing, "lga_rent"]
rental_clean.loc[missing, "infilling_method"] = "lga"

# Proximity fallback using original published suburb rents only
from scipy.spatial import cKDTree

points = suburbs[["suburb_id", "geometry"]].to_crs(7855).copy()
points["geometry"] = points.geometry.representative_point()
points["x"] = points.geometry.x
points["y"] = points.geometry.y

rental_clean = rental_clean.merge(
    points[["suburb_id", "x", "y"]],
    on="suburb_id",
    how="left"
)

for dwelling_type in rental_clean["dwelling_type"].unique():

    donors = rental_clean[
        (rental_clean["dwelling_type"] == dwelling_type)
        & (rental_clean["infilling_method"] == "suburb")
    ].dropna(subset=["x", "y"])

    targets = rental_clean[
        (rental_clean["dwelling_type"] == dwelling_type)
        & rental_clean["median_weekly_rent"].isna()
    ].dropna(subset=["x", "y"])

    if donors.empty or targets.empty:
        continue

    tree = cKDTree(donors[["x", "y"]].to_numpy())
    _, idx = tree.query(targets[["x", "y"]].to_numpy())

    rental_clean.loc[targets.index, "median_weekly_rent"] = (
        donors.iloc[idx]["median_weekly_rent"].to_numpy()
    )
    rental_clean.loc[targets.index, "infilling_method"] = "proximity"

# Remove temporary columns
rental_clean = rental_clean.drop(
    columns=["suburb_id", "lga_rent", "x", "y"]
)


In [74]:
# Infilling method breakdown - Greater Melbourne only

melbourne_rentals = rental_clean[
    rental_clean["suburb_join"].isin(suburbs["suburb_name_join"])
]

print("Rental records by infilling method:")
print(melbourne_rentals["infilling_method"].value_counts(dropna=False))

print("\nUnique suburbs by infilling method:")
print(
    melbourne_rentals.groupby("infilling_method")["suburb_join"].nunique()
)

print("\nTotal Greater Melbourne suburbs covered:",
      melbourne_rentals.loc[
          melbourne_rentals["median_weekly_rent"].notna(),
          "suburb_join"
      ].nunique(),
      "/", suburbs["suburb_id"].nunique()
)

Rental records by infilling method:
infilling_method
lga          2801
suburb       1130
proximity      73
Name: count, dtype: int64

Unique suburbs by infilling method:
infilling_method
lga          420
proximity     58
suburb       165
Name: suburb_join, dtype: int64

Total Greater Melbourne suburbs covered: 572 / 572


In [75]:
# 6 - Vicmap community facilities and schools
# Keep the types needed for school proximity and community infrastructure.
facilities = ds_06.to_crs(7855)
facilities = facilities[facilities.intersects(melbourne_shape)]
facilities = facilities[facilities["FTYPE"].isin([
    "education centre", "community venue", "community space", "cultural centre"
])]

# Education centres also include TAFEs, so limit them to school types.
school_types = ["primary school", "secondary school", "primary/secondary school", "special school"]
facilities = facilities[(facilities["FTYPE"] != "education centre") |
                       (facilities["FEATSUBTYP"].isin(school_types))]

vicmap_facilities = facilities[["FEATURE_ID", "NAME", "FTYPE", "FEATSUBTYP", "geometry"]].copy()
vicmap_facilities = vicmap_facilities.rename(columns={
    "FEATURE_ID": "facility_id", "NAME": "facility_name",
    "FTYPE": "facility_type", "FEATSUBTYP": "facility_subtype"
}).drop_duplicates(subset="facility_id")
vicmap_facilities = vicmap_facilities.to_crs(4326).reset_index(drop=True)

print("Relevant Vicmap facilities:", len(vicmap_facilities))
display(vicmap_facilities.head())

Relevant Vicmap facilities: 3233


,facility_id,facility_name,facility_type,facility_subtype,geometry
0,633059,ROCKBANK COMMUNITY,community venue,hall,MULTIPOINT ((144.65377 -37.72685))
1,633089,TOOLERN VALE MECHANICS HALL,community venue,hall,MULTIPOINT ((144.59377 -37.60772))
2,981488,BANCHORY COMMUNITY CENTRE,community venue,hall,MULTIPOINT ((144.74994 -37.69106))
3,981489,BLACKWOOD DRIVE COMMUNITY HALL,community venue,hall,MULTIPOINT ((144.5731 -37.69188))
4,981492,BROOKSIDE COMMUNITY CENTRE,community venue,community centre,MULTIPOINT ((144.73707 -37.74662))


In [76]:
# 7 - Public transport stops
# Keep the stop ID, name, transport type and location.
stops = ds_07.to_crs(7855)
stops = stops[stops.intersects(melbourne_shape)]
stops = stops[["STOP_ID", "STOP_NAME", "MODE", "geometry"]].copy()
stops = stops.rename(columns={"STOP_ID": "stop_id", "STOP_NAME": "stop_name", "MODE": "transport_mode"})
stops["transport_mode"] = stops["transport_mode"].fillna("").str.lower()
stops = stops[stops["transport_mode"].str.contains("bus|tram|train")]
transport_stops = stops.drop_duplicates(subset="stop_id").to_crs(4326).reset_index(drop=True)

print("Relevant transport stops:", len(transport_stops))
display(transport_stops.head())

Relevant transport stops: 23779


,stop_id,stop_name,transport_mode,geometry
0,11212,Flinders Street Station,regional train,POINT (144.96627 -37.81809)
1,11213,Flinders Street Station,regional train,POINT (144.96649 -37.81814)
2,11214,Flinders Street Station,regional train,POINT (144.96652 -37.8182)
3,11215,Flinders Street Station,regional train,POINT (144.96656 -37.81829)
4,11216,Flinders Street Station,regional train,POINT (144.9666 -37.81833)


In [77]:
# 10 - GPs, pharmacies and hospitals
# Combine the three layers into one table with a facility type.
health_parts = []

for service, table in ds_10.items():
    health = table.to_crs(7855)
    health = health[health.intersects(melbourne_shape)].copy()
    name_col = next((c for c in health.columns
                     if c.lower() in ["name", "facility_name", "service_name", "organisation_name"]), None)

    if name_col is None:
        health["facility_name"] = pd.NA
    else:
        health = health.rename(columns={name_col: "facility_name"})

    health["facility_type"] = service.replace("general_practices", "gp").replace("hospitals", "hospital").replace("pharmacies", "pharmacy")
    health_parts.append(health[["facility_type", "facility_name", "geometry"]])

health_services = gpd.GeoDataFrame(pd.concat(health_parts, ignore_index=True),
                                   geometry="geometry", crs=7855)
health_services = health_services.drop_duplicates().to_crs(4326)

print("Health service locations:", len(health_services))
display(health_services.head())

Health service locations: 2580


,facility_type,facility_name,geometry
0,gp,Hillcrest Health Centre,POINT (144.9143 -37.68442)
1,gp,Coolaroo Clinic,POINT (144.93155 -37.66459)
2,gp,Beachside Doctors,POINT (144.96656 -38.33221)
3,gp,Peak Medical Gpc,POINT (144.7038 -37.88137)
4,gp,Belgrave Medical Clinic,POINT (145.34709 -37.90715)


In [78]:
clean_tables = {
    "suburbs": suburbs,
    "census_people": census_people,
    "language_counts": language_counts,
    "rental_clean": rental_clean,
    "crime_clean": crime_clean,
    "vicmap_facilities": vicmap_facilities,
    "transport_stops": transport_stops,
    "health_services": health_services,
    "rental_availability": rental_availability}

In [79]:
# Export the data if needed 


for name, df in clean_tables.items():
    file = cleaned / f"{name}.csv"
    df.to_csv(file, index=False)
    print(f"Saved: {file}")

Saved: cleaned\suburbs.csv
Saved: cleaned\census_people.csv
Saved: cleaned\language_counts.csv
Saved: cleaned\rental_clean.csv
Saved: cleaned\crime_clean.csv
Saved: cleaned\vicmap_facilities.csv
Saved: cleaned\transport_stops.csv
Saved: cleaned\health_services.csv
Saved: cleaned\rental_availability.csv


## Data quality


In [80]:
# Test all cleaned datasets before building the OBT

# Tables and their expected columns
tables = {
    "Suburbs": (suburbs, ["suburb_id", "suburb_name", "suburb_name_join", "suburb_area_km2"]),
    "Census": (census_people, ["suburb_id", "population", "median_age"]),
    "Languages": (language_counts, ["suburb_id", "language", "speaker_count"]),
    "Rental": (rental_clean, ["suburb", "suburb_join", "median_weekly_rent", "dwelling_type", "period", "precinct"]),
    "Crime": (crime_clean, ["suburb", "suburb_join", "total_incidents", "person_incidents", "property_incidents"]),
    "Vicmap": (vicmap_facilities, ["facility_id", "facility_type", "facility_subtype"]),
    "Transport": (transport_stops, ["stop_id", "transport_mode"]),
    "Healthcare": (health_services, ["facility_type", "facility_name"]),
    "Rental Availability": (rental_availability, ["suburb_id", "dwelling_type", "5yr_avg_new_rentals"])
}

results = []

for name, (df, columns) in tables.items():

    # Check the table contains records
    has_rows = len(df) > 0

    # Check all required columns exist
    has_columns = all(col in df.columns for col in columns)

    # Check for missing values in required columns
    missing_values = df[columns].isna().sum().sum() if has_columns else 0

    # Check spatial datasets have geometry and a coordinate system
    spatial_ok = True

    if isinstance(df, gpd.GeoDataFrame):
        spatial_ok = (
            df.crs is not None
            and df.geometry.notna().all()
            and df.crs.to_epsg() == 4326
        )

    # Determine result
    passed = has_rows and has_columns and spatial_ok

    # Additional checks for the replacement language table only.
    if name == "Languages" and has_columns:
        passed = passed and (
            missing_values == 0
            and pd.api.types.is_numeric_dtype(df["speaker_count"])
            and df["speaker_count"].ge(0).all()
            and not df.duplicated(["suburb_id", "language"]).any()
        )

    results.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": missing_values,
        "Status": "PASS" if passed else "FAIL"
    })

# Display results
test_results = pd.DataFrame(results)

print(test_results)

print("Tests passed:", (test_results["Status"] == "PASS").sum(), "/", len(test_results))

               Dataset    Rows  Columns  Missing Values Status
0              Suburbs     572        5               0   PASS
1               Census     572        3               0   PASS
2            Languages  287716        3               0   PASS
3               Rental    4004        7            5698   PASS
4                Crime     648        7               0   PASS
5               Vicmap    3233        5               0   PASS
6            Transport   23779        4               0   PASS
7           Healthcare    2580        3               0   PASS
8  Rental Availability    3432        4               0   PASS
Tests passed: 9 / 9


In [81]:

# Tables that join back to suburbs on a shared key column
suburb_joins = {
    "census_people": ("suburb_id", "suburb_id"),
    "language_counts": ("suburb_id", "suburb_id"),
    "crime_clean": ("suburb_name_join", "suburb_join"),
    "rental_clean": ("suburb_name_join", "suburb_join"),
    "rental_availability": ("suburb_id", "suburb_id"),
}

# Tables with no suburb key at all - these join to suburbs spatially (point-in-polygon)
spatial_joins = ["vicmap_facilities", "transport_stops", "health_services"]

results = []
for name, (suburbs_col, child_col) in suburb_joins.items():
    child = clean_tables[name]

    # Only count suburbs with an available rental activity value
    if name == "rental_availability":
        child = child.dropna(subset=["5yr_avg_new_rentals"])

    suburb_keys = set(suburbs[suburbs_col])
    child_keys = set(child[child_col].dropna())

    # Coverage: how many suburbs get a match in the child table
    suburbs_matched = suburbs[suburbs_col].isin(child_keys).sum()

    # Accuracy: how many child keys match a real suburb
    child_matched = len(child_keys & suburb_keys)

    results.append({
        "Table": name,
        "Join": f"suburbs.{suburbs_col} = {name}.{child_col}",
        "Suburbs covered": suburbs_matched,
        "Suburbs total": len(suburbs),
        "Coverage": f"{suburbs_matched / len(suburbs):.1%}",
        "Child matched": child_matched,
        "Child total": len(child_keys),
        "Accuracy": f"{child_matched / len(child_keys):.1%}" if child_keys else "n/a",
    })

# Same two checks, done spatially: which suburbs contain at least one point, and which
# points actually fall inside one of our suburb boundaries at all.
for name in spatial_joins:
    child = clean_tables[name]
    joined = gpd.sjoin(
        child[["geometry"]].reset_index(drop=True),
        suburbs[["suburb_id", "geometry"]],
        how="left", predicate="within",
    )
    suburbs_matched = suburbs["suburb_id"].isin(joined["suburb_id"].dropna()).sum()
    child_matched = joined["suburb_id"].notna().sum()

    results.append({
        "Table": name,
        "Join": "point-in-polygon on geometry",
        "Suburbs covered": suburbs_matched,
        "Suburbs total": len(suburbs),
        "Coverage": f"{suburbs_matched / len(suburbs):.1%}",
        "Child matched": child_matched,
        "Child total": len(child),
        "Accuracy": f"{child_matched / len(child):.1%}" if len(child) else "n/a",
    })

join_quality = pd.DataFrame(results)
display(join_quality)


,Table,Join,Suburbs covered,Suburbs total,Coverage,Child matched,Child total,Accuracy
0,census_people,suburbs.suburb_id = census_people.suburb_id,572,572,100.0%,572,572,100.0%
1,language_counts,suburbs.suburb_id = language_counts.suburb_id,572,572,100.0%,572,572,100.0%
2,crime_clean,suburbs.suburb_name_join = crime_clean.suburb_...,569,572,99.5%,569,569,100.0%
3,rental_clean,suburbs.suburb_name_join = rental_clean.suburb...,572,572,100.0%,572,572,100.0%
4,rental_availability,suburbs.suburb_id = rental_availability.suburb_id,572,572,100.0%,572,572,100.0%
5,vicmap_facilities,point-in-polygon on geometry,482,572,84.3%,3096,3233,95.8%
6,transport_stops,point-in-polygon on geometry,466,572,81.5%,23777,23779,100.0%
7,health_services,point-in-polygon on geometry,362,572,63.3%,2579,2580,100.0%
